In [1]:
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from typing import TypedDict

d:\python\langchain_update\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatOllama(
    model = "llama3.2:latest"
)

In [12]:
class BlogState(TypedDict):
    title : str
    outline:str
    blog_content:str
    rating: int

In [ ]:
def create_outline(state: BlogState) -> BlogState :
    # fetch the title
    title = state["title"]

    # call llm to generate outline
    prompt = f"Generate a detailed outline for blog on the topic : {title}"
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline
    
    return state

In [ ]:
def create_blog(state : BlogState)-> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed on blog on title - {title} using the following outline - \n{outline}"
    blog_content = model.invoke(prompt).content

    state['blog_content'] = blog_content

    return state


In [17]:
def evaluate_blog(state: BlogState) -> BlogState:
    outline = state['outline']
    blog_content = state['blog_content']

    prompt = f"Based on the outline {outline} and blog content {blog_content} rate the blog out of 5."
    rating = model.invoke(prompt).content
    state['rating'] = rating

    return state

In [14]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)


workflow= graph.compile()

In [10]:
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state) 
print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here is a detailed outline for a blog on the topic "The Rise of AI in India":\n\n**Title:** "The Rise of AI in India: Trends, Opportunities, and Challenges"\n\n**I. Introduction**\n\n* Brief overview of AI and its growing presence in India\n* Importance of AI in India\'s economic and social development\n* Thesis statement: AI is rapidly transforming India\'s economy, society, and politics, presenting both opportunities and challenges.\n\n**II. Current State of AI in India**\n\n* Overview of India\'s AI ecosystem: startups, research institutions, and government initiatives\n* Analysis of AI adoption rates in various industries (e.g. healthcare, finance, education)\n* Discussion of the role of AI in addressing India\'s pressing issues (e.g. poverty, inequality, climate change)\n\n**III. Emerging Trends in AI in India**\n\n* Discussion of the latest trends in AI in India, such as:\n + AI for social good (e.g. education, healthcare, environmenta

In [ ]:
inital_state1 = {'title': 'Raise of AI in India'}
final_state1 = workflow.invoke(inital_state1)
print(final_state1)

In [ ]:
print(final_state1['rating'])